# Reference: parkway milepost points
Downloads the NPS Blue Ridge Parkway milepost layer (one point per whole mile) and stores it as
`brp.silver.mileposts`. The gold layer uses these points to place each road section on a map.

**Reference data:** the road's geometry rarely changes, so this notebook is run once (and re-run
only if NPS updates the layer). It is **not** part of the daily job.

**Steps**
1. **Settings:** imports, source URL, table names
2. **Download:** query the ArcGIS layer as GeoJSON (with paging)
3. **Save raw:** write the untouched GeoJSON to the bronze volume
4. **Transform:** one row per milepost: milepost, longitude, latitude
5. **Quality checks:** count, range, duplicates, location, and distance between neighbours
6. **Write:** replace the whole table
7. **Review:** table and map

In [0]:
# Standard library
import os
import json
import math
from datetime import datetime, timezone

# Third-party
import requests
import pandas as pd
from pyspark.sql.types import StructType, StructField, DoubleType, TimestampType

# ArcGIS query endpoint for the NPS milepost layer (layer 0 of the service)
MILEPOSTS_URL = (
    "https://services1.arcgis.com/fBc8EJBxQRMcHlei/arcgis/rest/services/"
    "SEAC_BLRI_Mileposts/FeatureServer/2/query"
)
MILEPOST_FIELD = "milepost"     # name of the milepost attribute in the layer
PAGE_SIZE = 1000                # the layer's max records per request

VOLUME_PATH = "/Volumes/brp/bronze/raw"
MILEPOSTS_TABLE = "brp.silver.mileposts"

MILEPOSTS_SCHEMA = StructType([
    StructField("milepost",   DoubleType()),
    StructField("longitude",  DoubleType()),
    StructField("latitude",   DoubleType()),
    StructField("fetched_at", TimestampType()),
])

In [0]:
# ---------------------------------------------------------------------------
# DOWNLOAD THE MILEPOST POINTS FROM ARCGIS
# The query asks for every feature (where=1=1), all attributes (outFields=*),
# as GeoJSON, in longitude/latitude (outSR=4326 = standard GPS coordinates).
# ArcGIS returns at most PAGE_SIZE features per request, so we ask page by
# page (resultOffset) until a page comes back short.
# ---------------------------------------------------------------------------

features = []   # every feature from every page
offset = 0      # how many features to skip (0 for the first page)

while True:
    response = requests.get(
        MILEPOSTS_URL,
        params={
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "outSR": 4326,
            "resultOffset": offset,
            "resultRecordCount": PAGE_SIZE,
        },
        timeout=60,
        headers={"User-Agent": "brp-closures-portfolio-project"},
    )
    response.raise_for_status()
    page = response.json()

    # ArcGIS reports many errors with status 200 and an "error" key in the
    # body, so raise_for_status() alone won't catch them
    if "error" in page:
        raise RuntimeError(f"ArcGIS error: {page['error']}")

    batch = page.get("features", [])
    features.extend(batch)

    if len(batch) < PAGE_SIZE:      # a short page means this was the last one
        break
    offset += PAGE_SIZE

fetched_at = datetime.now(timezone.utc)

print(len(features), "features downloaded")
# Show the attribute names of the first feature, to confirm MILEPOST_FIELD is right
print("Attribute fields:", sorted(features[0]["properties"]))
print("Example feature:", features[0])

In [0]:
fetched_at

In [0]:
# ---------------------------------------------------------------------------
# SAVE THE RAW DOWNLOAD (BRONZE COPY)
# The untouched features are saved as one GeoJSON file, so silver can be
# rebuilt later without downloading again. Dated folder = one copy per fetch.
# ---------------------------------------------------------------------------

fetch_date = fetched_at.date().isoformat()
folder = f"{VOLUME_PATH}/mileposts/{fetch_date}"
os.makedirs(folder, exist_ok=True)

raw_path = f"{folder}/mileposts.geojson"
with open(raw_path, "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)

print("Saved raw GeoJSON to:", raw_path)

In [0]:
features[0]

In [0]:
# ---------------------------------------------------------------------------
# TRANSFORM: ONE ROW PER MILEPOST
# Each GeoJSON feature looks like:
#   {"geometry": {"type": "Point", "coordinates": [longitude, latitude]},
#    "properties": {"milepost": 63, ...}}
# Note: GeoJSON puts LONGITUDE FIRST, then latitude.
# Every feature becomes either a row or a "skipped" entry, never lost.
# ---------------------------------------------------------------------------

rows = []      # good points
skipped = []   # features that couldn't be used (reported in the checks)

for feature in features:
    props = feature.get("properties") or {}
    geom = feature.get("geometry") or {}

    try:
        milepost = float(props[MILEPOST_FIELD])
        longitude, latitude = geom["coordinates"][:2]
        if geom.get("type") != "Point":
            raise ValueError("not a point")
    except (KeyError, TypeError, ValueError) as error:
        skipped.append((feature, str(error)))
        continue

    rows.append({
        "milepost":   milepost,
        "longitude":  float(longitude),
        "latitude":   float(latitude),
        "fetched_at": fetched_at,
    })

# Sort north to south (milepost 0 is the northern end in Virginia)
mileposts_df = pd.DataFrame(rows).sort_values("milepost").reset_index(drop=True)

print(f"Features: {len(features)} | Points: {len(rows)} | Skipped: {len(skipped)}")
for feature, reason in skipped[:10]:
    print("  SKIPPED:", reason, feature.get("properties"))
display(mileposts_df)

In [0]:
# ---------------------------------------------------------------------------
# QUALITY CHECKS
#   FAIL (assert) -> stop: don't save bad geometry
#   WARN (print)  -> saved, but worth a look
# The bounding box comes from the layer's published extent (data.gov),
# padded slightly.
# ---------------------------------------------------------------------------

MIN_LON, MAX_LON = -83.4, -78.7     # west / east limits of the parkway
MIN_LAT, MAX_LAT = 35.2, 38.1       # south / north limits
MAX_NEIGHBOUR_MI = 3.0              # straight-line distance between consecutive mileposts


def distance_miles(lon1, lat1, lon2, lat2):
    """Straight-line distance between two points on Earth (haversine formula)."""
    r = 3958.8                                           # Earth's radius in miles
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

# Self-test: one degree of latitude is about 69 miles
assert 68 < distance_miles(-80, 36, -80, 37) < 70, "Self-test: haversine distance"


# Known source error: two points labelled 89, none labelled 88.
# The "89" closer to milepost 87 is really 88.
is_89 = mileposts_df["milepost"] == 89
if is_89.sum() == 2 and not (mileposts_df["milepost"] == 88).any():
    mp87 = mileposts_df[mileposts_df["milepost"] == 87].iloc[0]
    dist = mileposts_df[is_89].apply(
        lambda p: distance_miles(mp87.longitude, mp87.latitude, p.longitude, p.latitude), axis=1)
    mileposts_df.loc[dist.idxmin(), "milepost"] = 88.0
    print("Corrected: relabelled one milepost 89 as 88")

mileposts_df = mileposts_df.sort_values("milepost").reset_index(drop=True)

# ---- FAIL checks ----------------------------------------------------------

# 1. Every feature was used (nothing skipped)
assert not skipped, f"Check 1: {len(skipped)} features skipped; see the list above"

# 2. About one point per mile for 469 miles
assert 450 <= len(mileposts_df) <= 480, f"Check 2: unexpected point count {len(mileposts_df)}"

# 3. Mileposts cover the whole parkway
assert mileposts_df["milepost"].min() <= 1, "Check 3: first milepost should be near 0"
assert mileposts_df["milepost"].max() >= 468, "Check 3: last milepost should be near 469"

# 4. No milepost appears twice
duplicates = mileposts_df[mileposts_df["milepost"].duplicated()]["milepost"].tolist()
assert not duplicates, f"Check 4: duplicate mileposts {duplicates}"

# 5. Every point is inside the parkway's area (catches swapped lon/lat)
outside = mileposts_df[
    ~mileposts_df["longitude"].between(MIN_LON, MAX_LON)
    | ~mileposts_df["latitude"].between(MIN_LAT, MAX_LAT)
]
assert outside.empty, f"Check 5: {len(outside)} points outside the parkway area:\n{outside}"

# 6. Consecutive mileposts are close together (catches misplaced points)
far_apart = []
points = mileposts_df.to_dict("records")
for prev, curr in zip(points, points[1:]):
    d = distance_miles(prev["longitude"], prev["latitude"], curr["longitude"], curr["latitude"])
    if d > MAX_NEIGHBOUR_MI:
        far_apart.append(f"MP {prev['milepost']} -> {curr['milepost']}: {d:.1f} mi")
for item in far_apart:
    print("Check 6 PROBLEM:", item)
assert not far_apart, f"Check 6: {len(far_apart)} neighbouring points over {MAX_NEIGHBOUR_MI} mi apart"

print("FAIL checks 1-6 passed")

# ---- WARN checks ----------------------------------------------------------

# 7. Missing whole mileposts (small gaps are fine: gold interpolates across them)
expected = set(range(0, 470))
present = set(mileposts_df["milepost"].round().astype(int))
missing = sorted(expected - present)
if missing:
    print(f"Check 7 WARNING: {len(missing)} whole mileposts missing: {missing[:20]}")

print("Quality checks complete (see any WARNING lines above)")

In [0]:
# ---------------------------------------------------------------------------
# WRITE: REPLACE THE WHOLE TABLE
# Reference data has no history: a new download fully replaces the old one.
# ---------------------------------------------------------------------------

# Align each row with MILEPOSTS_SCHEMA: pick the schema's columns by name, in schema order
rows_out = [tuple(r[field.name] for field in MILEPOSTS_SCHEMA.fields)
            for r in mileposts_df.to_dict("records")]
spark_df = spark.createDataFrame(rows_out, schema=MILEPOSTS_SCHEMA)

spark_df.write.mode("overwrite").saveAsTable(MILEPOSTS_TABLE)
print(f"Wrote {spark_df.count()} mileposts to {MILEPOSTS_TABLE}")

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- REVIEW: ALL MILEPOST POINTS, NORTH TO SOUTH
-- Checks that the points trace the parkway from Rockfish Gap, VA (MP 0)
-- to Cherokee, NC (MP 469).
--
-- To see the points on a map:
--   1. Run this cell
--   2. Under the result table, click "+" -> "Visualization"
--   3. Choose a map type (e.g. "Map (Markers)")
--   4. Set Latitude = latitude, Longitude = longitude
--   5. Optional: show milepost as the label/tooltip, so hovering a dot shows its number
--   6. Click "Save"; the map is stored with the notebook as a tab next to "Table"
--
-- What to look for:
--   - one continuous line of dots running southwest from Virginia into North Carolina
--   - no dots far away from the line (a misplaced point)
--   - no dots in the ocean or another continent (longitude/latitude swapped)
-- ---------------------------------------------------------------------------
SELECT milepost, latitude, longitude
FROM brp.silver.mileposts
ORDER BY milepost

Databricks visualization. Run in Databricks to view.